# Datathon IV — Predicción de ventas con MLOps

**Panadería Salvador · Artículo 3960 (BOLLERÍA)**

Máster en Data Analytics y Data Science — Code Space Academy

---

Panadería Salvador produce cada día sin saber cuánto va a vender. Pasarse genera merma:
producto que acaba en la basura. Quedarse corto es venta perdida. Este notebook construye
un modelo que estima las unidades que se venderán de un artículo en una fecha concreta, y
lo lleva hasta producción: seguimiento en MLflow, registro, despliegue como API REST y
escritura de las predicciones en la base de datos.

### Recorrido

| Sección | Qué se hace |
|---|---|
| 1 | Entorno y configuración |
| 2 | Carga de los datos |
| 3 | Transformadores: calendario, meteorología e histórico |
| 4 | Pipeline de scikit-learn |
| 5 | Partición temporal en train y test |
| 6 | Servidor de MLflow |
| 7 | Entrenamiento y evaluación dentro de un run |
| 8 | El problema de los retardos en el conjunto de test |
| 9 | Búsqueda de hiperparámetros |
| 10 | Registro del modelo |
| 11 | Despliegue como API REST |
| 12 | Predicciones contra la API |
| 13 | Escritura de los resultados en la base de datos |
| 14 | Conclusiones |

## 1. Entorno y configuración

Las credenciales se leen de variables de entorno mediante un fichero `.env` que **no está
en el repositorio**. La plantilla original del ejercicio traía el usuario y la contraseña
escritos en una celda; aquí no aparece ningún valor sensible en el código.

In [ ]:
import warnings

warnings.filterwarnings('ignore')

In [ ]:
import json
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from dotenv import load_dotenv

# Los transformadores viven en src/ para que MLflow pueda servirlos (ver sección 3)
sys.path.append(str(Path.cwd().parent / 'src'))

load_dotenv(Path.cwd().parent / '.env')

sns.set_theme(style='whitegrid')
pd.options.display.max_columns = None

In [ ]:
# Artículo a predecir
PRODUCT_ID = 3960
PRODUCT_FAMILY = 'BOLLERIA'

# Coordenadas de la panadería, para el parte meteorológico
LOCATION = (36.721477644071705, -4.363132134392174)

# Primera fecha del conjunto de test
HORIZONTE = datetime(2023, 4, 19)

# Retardos: 6 días y 4 semanas anteriores
DAILY_LAG, WEEKLY_LAG = 6, 4

DATA_DIR = Path.cwd().parent / 'data'
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5000')
MLFLOW_EXPERIMENT_NAME = 'Datathon IV'
MODEL_NAME = 'datathon_iv'
RANDOM_STATE = 42

print(f'Artículo {PRODUCT_ID} ({PRODUCT_FAMILY})')
print(f'Test a partir de {HORIZONTE.date()}')

## 2. Carga de los datos

Los datos salen de la vista `ventas_diarias_estudio`, que agrega las ventas horarias a
nivel de día y se queda con los cinco artículos de mayor facturación de cada familia.

El notebook intenta primero la base de datos. Si no hay conexión, usa los CSV locales que
genera `src/prepare_data.py`, que reconstruye esa misma vista desde los Excel de origen.
Así el proyecto se puede ejecutar sin credenciales.

In [ ]:
QUERY = """
    SELECT familia,
           articulo,
           fechaVenta AS fecha_venta,
           festivo,
           precio,
           cantidad,
           importe,
           orden_articulo_familia
    FROM ventas_diarias_estudio
"""


def cargar_desde_base_de_datos() -> pd.DataFrame:
    from database import DatabaseConnection
    return DatabaseConnection().query_to_df(QUERY)


def cargar_desde_csv() -> pd.DataFrame:
    train = pd.read_csv(DATA_DIR / 'train.csv')
    test = pd.read_csv(DATA_DIR / 'test.csv')
    return pd.concat([train, test], ignore_index=True)


try:
    df = cargar_desde_base_de_datos()
    print('Datos cargados desde la base de datos')
except Exception as e:
    df = cargar_desde_csv()
    print(f'Sin conexión a la base de datos ({type(e).__name__}). Usando los CSV locales.')

df['fecha_venta'] = pd.to_datetime(df['fecha_venta'])
df['articulo'] = df['articulo'].astype(int)
print(f'{df.shape[0]} filas, {df["articulo"].nunique()} artículos')
df.head()

### Filtrado por artículo

Se trabaja con el 3960, el de mayor facturación de BOLLERÍA. El mismo notebook sirve para
cualquier otro cambiando `PRODUCT_ID`.

In [ ]:
item_df = (df[df['articulo'] == PRODUCT_ID]
           .sort_values('fecha_venta')
           .reset_index(drop=True))

print(f'{len(item_df)} días, de {item_df["fecha_venta"].min().date()} '
      f'a {item_df["fecha_venta"].max().date()}')
item_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(item_df['fecha_venta'], item_df['cantidad'], linewidth=0.9)
ax.axvline(HORIZONTE, color='crimson', linestyle='--', label='Inicio del test')
ax.set_title(f'Unidades vendidas al día — artículo {PRODUCT_ID}')
ax.set_ylabel('Unidades')
ax.legend()
plt.tight_layout()
plt.show()

print(item_df['cantidad'].describe().round(1))

La serie tiene un ciclo semanal claro y bastante ruido diario. Eso condiciona el diseño
de las variables: el histórico reciente no basta, hace falta también lo que pasó el mismo
día de semanas anteriores.

In [ ]:
# Ciclo semanal
media_dia = (item_df.assign(dia=item_df['fecha_venta'].dt.dayofweek + 1)
             .groupby('dia')['cantidad'].mean())

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom'], media_dia.values,
       color='steelblue')
ax.set_title('Media de unidades por día de la semana')
ax.set_ylabel('Unidades')
plt.tight_layout()
plt.show()

## 3. Transformadores

Tres bloques de información, cada uno con su transformador:

| Bloque | Transformador | Aporta |
|---|---|---|
| Calendario | `CalendarTransformer` | día, mes, año, día de la semana, fin de semana, festivo |
| Meteorología | `WeatherTransformer` | temperatura media, mínima y máxima, y si llovió |
| Histórico | `ToSupervisedTransformer` | ventas de los 6 días y las 4 semanas anteriores |

### Por qué están en `src/transformers.py` y no aquí

Esta decisión es la que hace que el despliegue funcione. Cuando MLflow sirve el modelo con
`mlflow models serve`, lo hace en **un proceso aparte** que tiene que deserializar el
pipeline. Si las clases estuvieran definidas en este notebook, quedarían registradas como
parte de `__main__` y ese proceso no sabría dónde encontrarlas: el servidor fallaría al
arrancar con un `ModuleNotFoundError`.

Al vivir en un módulo, basta con pasar `code_paths` al registrar el modelo para que MLflow
empaquete el fichero junto a él.

### Dos correcciones sobre la plantilla

1. **Las fechas se convierten con `pd.to_datetime` antes de extraer día y mes.** La versión
   original usaba el accesor `.dt` directamente. Al llamar a la API REST, las fechas viajan
   como texto dentro del JSON, porque JSON no tiene tipo fecha: con el accesor directo, el
   despliegue reventaría.
2. **El cruce con los datos meteorológicos es `left`, no `inner`.** Con un `inner`,
   cualquier fecha que meteostat no tuviera desaparecería y el pipeline devolvería menos
   filas de las que recibió, desalineándose con el target. Con `left` se conservan todas y
   los huecos los rellena el imputador.

In [ ]:
from transformers import (CalendarTransformer, ToSupervisedTransformer,
                          WeatherTransformer, lag_feature_names)

calendar_transformer = CalendarTransformer()
weather_transformer = WeatherTransformer(LOCATION)
to_supervised_transformer = ToSupervisedTransformer(DAILY_LAG, WEEKLY_LAG)

calendar_transformer.transform(item_df).head()

In [ ]:
weather_transformer.transform(item_df).head()

In [ ]:
to_supervised_transformer.transform(item_df).head(8)

Las primeras filas tienen huecos: no hay 28 días anteriores para el primer día de la
serie. El imputador del pipeline se encarga de ellos. Este detalle vuelve a aparecer en la
sección 8, donde resulta ser más importante de lo que parece.

## 4. Pipeline

```
TransformedTargetRegressor
└── Pipeline
    ├── FeatureUnion ......... calendario + meteorología + retardos
    ├── ColumnTransformer .... OneHotEncoder (categóricas) + MinMaxScaler (numéricas)
    ├── SimpleImputer ........ rellena los retardos sin histórico
    └── SVR .................. el modelo
```

Todo el preprocesamiento va **dentro** del pipeline. No es cosmético: hace que el escalado
y la codificación se ajusten solo con los datos de entrenamiento de cada partición, y que
el objeto desplegado sepa transformar los datos crudos por sí mismo. La API REST recibe las
columnas tal y como salen de la base de datos.

In [ ]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.svm import SVR

# verbose_feature_names_out=False mantiene los nombres de columna tal cual
# ('day', 'tavg', ...). Con el prefijo del transformador por delante, el
# ColumnTransformer de abajo no encontraría las columnas que declara.
preprocessor = FeatureUnion(transformer_list=[
    ('calendar_transformer', calendar_transformer),
    ('weather_transformer', weather_transformer),
    ('to_supervised_transformer', to_supervised_transformer),
], verbose_feature_names_out=False)

categorical_features = ['day', 'month', 'year', 'dayofweek',
                        'isWeekend', 'isHoliday', 'isRaining']
numerical_features = ['tavg', 'tmin', 'tmax'] + lag_feature_names(DAILY_LAG, WEEKLY_LAG)

data_scaler = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(sparse_output=False, handle_unknown='ignore'),
     categorical_features),
    ('scaler', MinMaxScaler(feature_range=(0, 1)), numerical_features),
])

print(f'{len(categorical_features)} categóricas, {len(numerical_features)} numéricas')

### El escalado del target

El SVR es sensible a la escala de la variable objetivo, así que también se escala, con
`TransformedTargetRegressor`.

Aquí hay otra corrección sobre la plantilla: esta usaba **la misma instancia** de
`MinMaxScaler` para las variables de entrada y para el target. Al ser el mismo objeto, el
segundo ajuste pisaba al primero y el escalado quedaba corrupto. Cada uno necesita su
propia instancia.

In [ ]:
def construir_pipeline(model=None) -> TransformedTargetRegressor:
    """Construye el pipeline completo con un modelo dado."""
    input_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('data_scaler', data_scaler),
        ('imputer', SimpleImputer(strategy='mean')),
        ('model', model if model is not None else SVR()),
    ])
    # Sin salida en pandas, el FeatureUnion devuelve un array y el
    # ColumnTransformer no puede seleccionar columnas por nombre.
    input_pipeline.set_output(transform='pandas')

    # Instancia propia para el target: no se reutiliza la de las entradas
    return TransformedTargetRegressor(regressor=input_pipeline,
                                      transformer=MinMaxScaler(feature_range=(0, 1)))


final_pipeline = construir_pipeline()
final_pipeline

## 5. Partición temporal

La partición es **cronológica**, no aleatoria: el test son los últimos días de la serie,
que el modelo no ve durante el entrenamiento. En una serie temporal, repartir los días al
azar entrenaría con días posteriores a los que luego se evalúan.

In [ ]:
df_train = item_df[item_df['fecha_venta'] < HORIZONTE].reset_index(drop=True)
df_test = item_df[item_df['fecha_venta'] >= HORIZONTE].reset_index(drop=True)

X_train, y_train = df_train, df_train['cantidad']
X_test, y_test = df_test, df_test['cantidad']

print(f'Train: {len(df_train)} días '
      f'({df_train["fecha_venta"].min().date()} a {df_train["fecha_venta"].max().date()})')
print(f'Test:  {len(df_test)} días '
      f'({df_test["fecha_venta"].min().date()} a {df_test["fecha_venta"].max().date()})')

El test son 12 días y no 30, pese a lo que dice la plantilla: la vista de estudio corta
en el 1 de mayo de 2023, así que entre el horizonte y el final de los datos solo quedan 12
jornadas. Es un conjunto pequeño, y eso conviene tenerlo presente al leer las métricas: una
sola jornada rara mueve mucho el resultado. Por eso la validación cruzada sobre el
entrenamiento es más informativa que el test en sí.

## 6. Servidor de MLflow

En una terminal aparte, desde la raíz del repositorio:

```bash
mlflow server --host 127.0.0.1 --port 5000 \
  --backend-store-uri sqlite:///mlflow.db \
  --default-artifact-root ./mlartifacts
```

El `--backend-store-uri` con SQLite no es opcional: el registro de modelos de la sección 10
necesita una base de datos, con el almacén de ficheros por defecto no funciona.

La interfaz queda en <http://localhost:5000>.

In [ ]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

# Autolog registra parámetros y métricas por su cuenta.
# log_models=False porque el modelo se registra a mano en la sección 10: hay que
# pasarle code_paths para que el servidor encuentre los transformadores.
mlflow.sklearn.autolog(log_models=False)

print(f'Conectado a {MLFLOW_TRACKING_URI}')
print(f'Experimento: {MLFLOW_EXPERIMENT_NAME}')

## 7. Entrenamiento y evaluación

### Por qué `TimeSeriesSplit` y no `KFold`

La plantilla validaba con `KFold(shuffle=True)`. En una serie temporal con variables de
retardo eso es una fuga: al barajar los días, el modelo se entrena con jornadas
**posteriores** a las que después evalúa, y además los retardos de una partición contienen
valores que están en la partición de validación. La métrica sale inflada y no dice nada
sobre cómo se comportará el modelo mañana.

`TimeSeriesSplit` respeta el orden: entrena siempre con el pasado y valida con el futuro
inmediato, que es exactamente la situación real.

### Las métricas

- **R²**: qué parte de la variabilidad explica el modelo.
- **MAE**: el error medio en unidades de producto. Es la que entiende el panadero.
- **RMSE**: penaliza más los errores grandes, los días en que la predicción se va mucho.

In [ ]:
from sklearn import metrics
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

cv = TimeSeriesSplit(n_splits=5)


def evaluar(y_true, y_pred) -> dict:
    """Métricas de regresión en unidades de producto."""
    return {
        'r2': metrics.r2_score(y_true, y_pred),
        'mae': metrics.mean_absolute_error(y_true, y_pred),
        'rmse': float(np.sqrt(metrics.mean_squared_error(y_true, y_pred))),
    }

In [ ]:
with mlflow.start_run(run_name='SVR_baseline') as run:

    mlflow.set_tag('product_id', PRODUCT_ID)
    mlflow.set_tag('product_family', PRODUCT_FAMILY)
    mlflow.set_tag('validacion', 'TimeSeriesSplit(5)')

    final_pipeline.fit(X_train, y_train)
    y_pred = final_pipeline.predict(X_test)

    r2_cross_val = cross_val_score(final_pipeline, X_train, y_train,
                                   cv=cv, scoring='r2')
    mlflow.log_metric('r2_cross_val', r2_cross_val.mean())
    mlflow.log_metric('r2_cross_val_std', r2_cross_val.std())

    resultados_test = evaluar(y_test, y_pred)
    for nombre, valor in resultados_test.items():
        mlflow.log_metric(f'{nombre}_test', valor)

    mlflow.log_param('daily_lag', DAILY_LAG)
    mlflow.log_param('weekly_lag', WEEKLY_LAG)

    run_id_baseline = run.info.run_id

print(f'R² validación cruzada: {r2_cross_val.mean():.4f} '
      f'(± {r2_cross_val.std():.4f})')
print(f'R² test:   {resultados_test["r2"]:.4f}')
print(f'MAE test:  {resultados_test["mae"]:.1f} unidades')
print(f'RMSE test: {resultados_test["rmse"]:.1f} unidades')

### Predicciones frente a realidad

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].scatter(y_pred, y_test, s=60, alpha=0.8)
lims = [min(y_pred.min(), y_test.min()) * 0.9, max(y_pred.max(), y_test.max()) * 1.1]
axes[0].plot(lims, lims, color='crimson', linestyle='--', label='Predicción perfecta')
axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Real')
axes[0].set_title('Predicción frente a realidad')
axes[0].legend()

fechas = df_test['fecha_venta'].dt.strftime('%d-%m')
axes[1].plot(fechas, y_test.values, marker='o', label='Real')
axes[1].plot(fechas, y_pred, marker='s', label='Predicción')
axes[1].set_title('Evolución en el periodo de test')
axes[1].set_ylabel('Unidades')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

plt.tight_layout()
plt.savefig('predicciones_baseline.png', dpi=110)
plt.show()

In [ ]:
# El gráfico se guarda como artefacto del run
with mlflow.start_run(run_id=run_id_baseline):
    mlflow.log_artifact('predicciones_baseline.png')
print('Gráfico registrado en el run', run_id_baseline)

## 8. El problema de los retardos en el conjunto de test

Esta sección corrige el fallo de diseño más serio de la plantilla.

`ToSupervisedTransformer` calcula los retardos **dentro del DataFrame que recibe**. Al
predecir, recibe solo las 12 filas del test. Y con 12 filas:

- los cuatro retardos semanales (7, 14, 21 y 28 días) son **todos nulos**, porque no hay
  tantos días en el DataFrame;
- los retardos diarios solo existen a partir de la séptima fila.

Esos nulos los rellena el imputador con la media del entrenamiento. Es decir: el modelo
predice el periodo de test **prácticamente sin histórico**, apoyándose casi solo en
calendario y meteorología. Y el histórico era justamente la información más valiosa.

Lo comprobamos.

In [ ]:
lags_test = to_supervised_transformer.transform(df_test)
nulos = lags_test.isna().mean().mul(100).round(1)

print('Porcentaje de valores nulos en los retardos del conjunto de test:')
print(nulos.to_string())

Los tres retardos semanales largos son nulos en el **100 %** de las filas, y el de una
semana en el 58 %. El modelo está prediciendo casi a ciegas respecto al histórico.

### La corrección

Para predecir el día *t* hacen falta las ventas reales de *t−1* a *t−28*, y esos días están
en el entrenamiento. La solución es pasar al pipeline los últimos 28 días de train
**por delante** del test, dejar que calcule los retardos sobre la serie continua, y quedarse
solo con las predicciones del tramo de test.

Esto no es un truco de evaluación: es exactamente lo que ocurre en producción. Para predecir
el martes, la panadería conoce las ventas de los días anteriores. La versión sin corregir
simula una situación que nunca se da, la de tener que predecir doce días seguidos sin saber
nada de lo vendido hasta ayer.

**Una advertencia sobre lo que mide la tabla siguiente.** Con solo 12 días de test, la
diferencia entre las dos versiones es pequeña y puede salir en cualquier dirección según el
tramo concreto que toque. El argumento para corregirlo no es que mejore la métrica, sino que
la versión sin corregir no representa el problema real: alimentar al modelo con la media del
entrenamiento en lugar de con el histórico es sencillamente incorrecto, salga el número que
salga en doce jornadas. La evidencia de peso es la tabla de nulos de arriba, no el R².

In [ ]:
def predecir_con_historial(pipeline, historico: pd.DataFrame,
                           futuro: pd.DataFrame, lag_dias: int = 28) -> np.ndarray:
    """Predice `futuro` anteponiendo los últimos días de `historico`.

    Los retardos se calculan sobre la serie continua, así que las filas del futuro
    disponen de su histórico real en lugar de valores imputados.

    Returns:
        Las predicciones correspondientes únicamente a las filas de `futuro`.
    """
    marco = pd.concat([historico.tail(lag_dias), futuro], ignore_index=True)
    return pipeline.predict(marco)[-len(futuro):]


y_pred_historial = predecir_con_historial(final_pipeline, df_train, df_test)
resultados_historial = evaluar(y_test, y_pred_historial)

comparativa = pd.DataFrame([
    {'enfoque': 'Test aislado (plantilla)', **resultados_test},
    {'enfoque': 'Con histórico (corregido)', **resultados_historial},
])
comparativa.round(4)

In [ ]:
with mlflow.start_run(run_name='SVR_con_historial'):
    mlflow.set_tag('product_id', PRODUCT_ID)
    mlflow.set_tag('product_family', PRODUCT_FAMILY)
    mlflow.set_tag('correccion', 'retardos calculados sobre la serie continua')
    for nombre, valor in resultados_historial.items():
        mlflow.log_metric(f'{nombre}_test', valor)

print('Diferencia en MAE: '
      f'{resultados_historial["mae"] - resultados_test["mae"]:+.1f} unidades')
print('(sobre 12 días: tómalo como orientativo, no como prueba)')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(fechas, y_test.values, marker='o', linewidth=2, label='Real')
ax.plot(fechas, y_pred, marker='s', linestyle='--', alpha=0.7,
        label='Test aislado')
ax.plot(fechas, y_pred_historial, marker='^', label='Con histórico')
ax.set_title('Efecto de calcular los retardos sobre la serie continua')
ax.set_ylabel('Unidades')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig('comparativa_historial.png', dpi=110)
plt.show()

## 9. Búsqueda de hiperparámetros

El SVR por defecto rara vez es el mejor. Se buscan `C`, `gamma` y `epsilon` con
`GridSearchCV`, usando otra vez `TimeSeriesSplit` para no romper el orden temporal.

`C` controla cuánto penaliza los errores, `gamma` la amplitud del kernel y `epsilon` el
margen dentro del cual un error no cuenta.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'regressor__model__C': [1, 10, 100],
    'regressor__model__gamma': ['scale', 0.01, 0.1],
    'regressor__model__epsilon': [0.01, 0.1],
}

busqueda = GridSearchCV(construir_pipeline(), param_grid, cv=cv,
                        scoring='r2', n_jobs=1, verbose=1)

with mlflow.start_run(run_name='SVR_gridsearch') as run_gs:
    mlflow.set_tag('product_id', PRODUCT_ID)
    mlflow.set_tag('product_family', PRODUCT_FAMILY)
    mlflow.set_tag('busqueda', 'GridSearchCV')

    busqueda.fit(X_train, y_train)

    mejor_pipeline = busqueda.best_estimator_
    y_pred_gs = predecir_con_historial(mejor_pipeline, df_train, df_test)
    resultados_gs = evaluar(y_test, y_pred_gs)

    mlflow.log_params(busqueda.best_params_)
    mlflow.log_metric('r2_cross_val', busqueda.best_score_)
    for nombre, valor in resultados_gs.items():
        mlflow.log_metric(f'{nombre}_test', valor)

    run_id_gs = run_gs.info.run_id

print('Mejores parámetros:', busqueda.best_params_)
print(f'R² validación cruzada: {busqueda.best_score_:.4f}')
print(f'R² test:   {resultados_gs["r2"]:.4f}')
print(f'MAE test:  {resultados_gs["mae"]:.1f} unidades')

In [ ]:
resumen = pd.DataFrame([
    {'modelo': 'SVR por defecto, test aislado', **resultados_test},
    {'modelo': 'SVR por defecto, con histórico', **resultados_historial},
    {'modelo': 'SVR ajustado, con histórico', **resultados_gs},
]).sort_values('mae')
resumen.round(4)

## 10. Registro del modelo

Se registra el mejor pipeline con un nombre y una versión, para poder desplegarlo sin tocar
el código que lo consume.

**`code_paths` es la clave del despliegue.** Le dice a MLflow que empaquete
`src/transformers.py` junto al modelo. Sin eso, el proceso que sirve el modelo no sabría
reconstruir `CalendarTransformer`, `WeatherTransformer` ni `ToSupervisedTransformer`, y
fallaría al arrancar.

El alias `production` permite referirse al modelo como `models:/datathon_iv@production`, sin
depender del número de versión.

In [ ]:
from mlflow import MlflowClient

SRC_PATH = str(Path.cwd().parent / 'src')

with mlflow.start_run(run_id=run_id_gs):
    info = mlflow.sklearn.log_model(
        mejor_pipeline,
        artifact_path='model',
        code_paths=[SRC_PATH],                 # ← sin esto el despliegue no arranca
        input_example=df_test.head(3),
        serialization_format='cloudpickle',
    )

version = mlflow.register_model(info.model_uri, MODEL_NAME)
MlflowClient().set_registered_model_alias(MODEL_NAME, 'production', version.version)

print(f'Modelo registrado: {MODEL_NAME} versión {version.version}')
print(f'Alias "production" apuntando a la versión {version.version}')

## 11. Despliegue como API REST

En una terminal aparte:

```bash
export MLFLOW_TRACKING_URI=http://localhost:5000
mlflow models serve -m "models:/datathon_iv@production" -p 5001 --env-manager local
```

Cuando aparezca algo así, está funcionando:

```
[INFO] Starting gunicorn 21.2.0
[INFO] Listening at: http://127.0.0.1:5001 (236041)
[INFO] Using worker: sync
[INFO] Booting worker with pid: 236048
```

El puerto tiene que ser distinto del 5000, que ocupa el servidor de seguimiento.

In [ ]:
URL = os.getenv('MODEL_SERVING_URI', 'http://127.0.0.1:5001/invocations')

try:
    estado = requests.get(URL.replace('/invocations', '/ping'), timeout=5)
    print(f'Servidor del modelo: {estado.status_code}')
except requests.exceptions.RequestException as e:
    print(f'El servidor del modelo no responde en {URL}')
    print(f'  ({type(e).__name__})')
    print('  Arráncalo con el comando de arriba antes de seguir.')

## 12. Predicciones contra la API

Se envían al modelo desplegado los mismos datos que se le pasarían en producción: los
últimos 28 días conocidos más las fechas a predecir. Las fechas viajan como texto porque
JSON no tiene tipo fecha; el transformador de calendario las convierte por su cuenta.

In [ ]:
def preparar_payload(historico: pd.DataFrame, futuro: pd.DataFrame,
                     lag_dias: int = 28) -> dict:
    """Arma el cuerpo de la petición en el formato que espera MLflow."""
    marco = pd.concat([historico.tail(lag_dias), futuro], ignore_index=True)
    marco = marco.copy()
    marco['fecha_venta'] = marco['fecha_venta'].astype(str)
    marco = marco.where(pd.notna(marco), None)
    return {
        'dataframe_split': {
            'columns': marco.columns.tolist(),
            'data': marco.values.tolist(),
        }
    }


payload = preparar_payload(df_train, df_test)
print(f'Se envían {len(payload["dataframe_split"]["data"])} filas '
      f'({len(df_test)} a predecir y 28 de histórico)')

In [ ]:
respuesta = requests.post(URL, headers={'Content-Type': 'application/json'},
                         data=json.dumps(payload), timeout=120)
respuesta.raise_for_status()

predictions = respuesta.json()['predictions'][-len(df_test):]

print(f'Status: {respuesta.status_code}')
print(f'{len(predictions)} predicciones recibidas')

# Comprobación: la API debe devolver lo mismo que el modelo local
locales = predecir_con_historial(mejor_pipeline, df_train, df_test)
print(f'Diferencia máxima con el modelo local: '
      f'{np.abs(np.array(predictions) - locales).max():.6f}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(fechas, y_test.values, marker='o', linewidth=2, label='Ventas reales')
ax.plot(fechas, predictions, marker='s', label='Predicción de la API')
ax.set_title(f'Predicción servida por la API REST — artículo {PRODUCT_ID}')
ax.set_ylabel('Unidades')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

## 13. Escritura de los resultados en la base de datos

Las predicciones se guardan en la base de datos para que otras herramientas —Power BI,
Tableau— puedan consumirlas. Es el último eslabón: el modelo deja de ser un notebook y pasa
a alimentar un sistema.

In [ ]:
df_prediccion = pd.DataFrame({
    'fecha': df_test['fecha_venta'].dt.strftime('%Y-%m-%d'),
    'cantidad': predictions,
    'articulo': PRODUCT_ID,
    'familia': PRODUCT_FAMILY,
})
df_prediccion.head()

In [ ]:
GROUP_NAME = os.getenv('GROUP_NAME', '')
DATABASE = os.getenv('DB_SANDBOX', 'sandbox')
table_name = f'Materials_Prediction_Group_{GROUP_NAME}'

try:
    from database import DatabaseConnection

    db = DatabaseConnection(database=DATABASE)
    db.insert_dataframe(df_prediccion, table_name)
    print(f'{len(df_prediccion)} predicciones escritas en {DATABASE}.{table_name}')

    comprobacion = db.query_to_df(f'SELECT * FROM {table_name}')
    print(f'Comprobación: la tabla tiene {len(comprobacion)} filas')
    display(comprobacion.head())
except Exception as e:
    print(f'No se pudo escribir en la base de datos: {type(e).__name__}')
    print(f'  {e}')
    df_prediccion.to_csv(DATA_DIR / 'predicciones.csv', index=False)
    print(f'  Guardadas en {DATA_DIR / "predicciones.csv"}')

## 14. Conclusiones

**El fallo más importante no estaba en el modelo, sino en cómo se le daban los datos.** Tal
como estaba la plantilla, al predecir el test los cuatro retardos semanales eran nulos en el
100 % de las filas y los rellenaba el imputador con la media: el modelo predecía casi sin
histórico, que era precisamente su mejor información. Calcular los retardos sobre la serie
continua lo corrige.

Conviene ser preciso sobre lo que eso demuestra: con 12 días de test, la diferencia de
métrica entre ambas versiones es pequeña y poco estable. La corrección no se justifica
porque suba el R², sino porque la versión original mide una situación que no existe. Para
saber cuánto vale de verdad haría falta un test mucho más largo.

**La validación importa tanto como el modelo.** Validar una serie temporal con
`KFold(shuffle=True)` entrena con días posteriores a los que evalúa y da una métrica que no
se parece a la realidad. `TimeSeriesSplit` da números más bajos y más honestos.

**El despliegue se decide al escribir el código, no al desplegarlo.** Tener los
transformadores en un módulo y pasar `code_paths` al registrar el modelo es lo que permite
que `mlflow models serve` arranque. Si las clases hubieran quedado en el notebook, el
servidor fallaría y no habría forma de arreglarlo sin reentrenar.

**El test es corto.** Doce días dan una métrica inestable: una jornada rara la mueve mucho.
Por eso el R² de validación cruzada sobre el entrenamiento es más fiable que el del test.
Con más histórico disponible, lo razonable sería una validación con ventana deslizante sobre
varios meses.

### Qué haría a continuación

- Probar modelos de boosting (XGBoost, LightGBM), que suelen rendir mejor que un SVR en
  tabular con variables de calendario.
- Añadir variables de promociones y de festivos locales de Málaga, que el calendario actual
  no distingue de los nacionales.
- Extender el pipeline a los 15 artículos de la vista de estudio y comparar si conviene un
  modelo por artículo o uno solo con el artículo como variable.
- Automatizar el reentrenamiento, para que el modelo no envejezca según cambian los hábitos
  de compra.